### Implementing HTTP PUT/GET 
GET request - User get the recipe from the server
PUT request - User send the feekback to the server

#### Client Pseudo-code

In [ ]:
function getRecipes(baseUrl, user, preferences, ingredients):
    qs = new QueryStringBuilder()
    qs.add("username", user.username)
    qs.add("email", user.email)
    for d in preferences.diet: qs.add("preferences.diet", d)
    for a in preferences.allergies: qs.add("preferences.allergies", a)
    for ing in ingredients:
        qs.add("ingredient_name", ing.name)
        # qty can be empty：send the empty string or do not sent this information
        if ing.qty_g is not null:
            qs.add("ingredient_qty_g", ing.qty_g)
        else:
            qs.add("ingredient_qty_g", "") 
    url = baseUrl + "/v1/recipes?" + qs.encode()

    req = new HttpRequest("GET", url)
    req.setHeader("Accept", "application/json")
    res = httpClient.send(req)

    if res.statusCode == 200:
        body = res.body
        recipes = JSON.parse(body)["recipes"]
        return recipes
    else:
        throw Error("HTTP " + res.statusCode + ": " + res.body)


### Server（GET /v1/recipes）Pseudo-code

In [ ]:
handler GET /v1/recipes:
    # 1) 解析 Query Params
    username = qp["username"]
    email = qp["email"]
    diet = qp.getAll("preferences.diet")      # key → list
    allergies = qp.getAll("preferences.allergies")
    names = qp.getAll("ingredient_name")
    qtys = qp.getAll("ingredient_qty_g")      # 與 names 以 index 對齊

    # 2) 構造 domain objects
    ingredients = []
    for i in range(0, len(names)):
        name = names[i]
        qty = parseNullableInt(qtys[i])  // 空字串 → null
        ingredients.append(Ingredient(name, qty))

    // 3) 業務邏輯：根據原料 + 偏好 生成食譜
    recipes = recipeService.generate(username, email, diet, allergies, ingredients)

    // 4) 序列化回傳
    responseJson = {
        "request_id": newRequestId(),
        "recipes": map(recipes, toJson)
    }
    return HttpResponse(200, JSON.stringify(responseJson), headers={"Content-Type":"application/json"})
